<a href="https://colab.research.google.com/github/MariiaOsokina/LLM_Architecture---Nebius-hometask/blob/main/MO_LLM_Architecture-homework-Week8-part1-lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework: LoRA from Scratch on GPT-2

In this assignment you will:

1. Implement **LoRA** (Low-Rank Adaptation) from scratch as a thin wrapper around `nn.Linear`.
2. Inject it into a frozen pre-trained **GPT-2 (small, 124M)** and verify only the LoRA params are trainable.
3. Fine-tune on **TinyShakespeare** so the model starts producing pseudo-Elizabethan text.
4. Save / load just the adapter (a few MB instead of 500MB).
5. Re-do the same fine-tune with the `peft` library and confirm the results match.
6. **Bonus:** swap the dataset for Rick & Morty dialogue, Hermione Granger lines, or Yoda quotes and watch the style transfer.

The whole thing fits on a free Colab T4 in well under an hour.

> **Submission:** run all cells, keep the printed sample outputs, and save the notebook. Written answers go in the `# YOUR ANSWER:` markdown cells.


## 0 · LoRA in one minute

For any frozen linear layer $W_0 \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}}$, LoRA adds a learned low-rank update:

$$
h = W_0 x + \Delta W \cdot x, \qquad \Delta W = \frac{\alpha}{r}\, B A
$$

where $A \in \mathbb{R}^{r \times d_{\text{in}}}$ and $B \in \mathbb{R}^{d_{\text{out}} \times r}$ with $r \ll \min(d_{\text{in}}, d_{\text{out}})$.

Key facts:

- $A$ is initialized with Kaiming uniform, $B$ with **zeros** ⇒ $\Delta W = 0$ at init, so the wrapped model behaves identically to the original on step 0.
- Only $A$ and $B$ are trained ⇒ the trainable param count drops by ~3 orders of magnitude.
- At inference you can either (a) keep $A, B$ separate, or (b) merge: $W \leftarrow W_0 + \frac{\alpha}{r} B A$ — same FLOPs as the original.
- $\alpha$ is a scaling hyperparameter; effective learning rate of the update scales with $\alpha/r$.

Reference: Hu et al., *LoRA: Low-Rank Adaptation of Large Language Models*, 2021 — https://arxiv.org/abs/2106.09685


## 1 · Setup

If you're on Colab, run the install cell. Local installs may already have these.


In [ ]:
# Colab install. Skip if you already have these locally.
!pip -q install "transformers>=4.40" "datasets>=2.18" "peft>=0.10" "accelerate>=0.27"

In [ ]:
import math
import os
import random
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import transformers
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device, "| transformers:", transformers.__version__)

device: cuda | transformers: 5.0.0


MO Note:
By setting a **SEED** (programmers often use 42 or 1234), you ensure that reproducibility. Every time you run your Colab notebook, the text will be shuffled and split in the exact same way, allowing you to accurately compare your experiments.

## 2 · Inspect GPT-2

We'll use **gpt2** (124M params). Let's load it and see what kinds of layers live inside.

> **Heads up:** HuggingFace's GPT-2 uses `transformers.pytorch_utils.Conv1D` (a quirky historical choice — it's just a linear layer with weight transposed) for `c_attn` and `c_proj`, **not** `nn.Linear`. That makes writing a generic LoRA wrapper annoying. We'll convert these to plain `nn.Linear` first so your LoRA code stays clean.


In [ ]:
MODEL_NAME = "gpt2"

tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

n_total = sum(p.numel() for p in model.parameters())
print(f"Total params: {n_total/1e6:.1f}M")

# Print one transformer block to see the layer types
print(model.transformer.h[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Total params: 124.4M
GPT2Block(
  (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (attn): GPT2Attention(
    (c_attn): Conv1D(nf=2304, nx=768)
    (c_proj): Conv1D(nf=768, nx=768)
    (attn_dropout): Dropout(p=0.1, inplace=False)
    (resid_dropout): Dropout(p=0.1, inplace=False)
  )
  (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (mlp): GPT2MLP(
    (c_fc): Conv1D(nf=3072, nx=768)
    (c_proj): Conv1D(nf=768, nx=3072)
    (act): NewGELUActivation()
    (dropout): Dropout(p=0.1, inplace=False)
  )
)


In [ ]:
# Helper: convert all transformers Conv1D modules to nn.Linear (functionally identical).
# This is given to you. Read it — but you don't need to modify it.
from transformers.pytorch_utils import Conv1D


def conv1d_to_linear(conv: Conv1D) -> nn.Linear:
    # Conv1D stores weight as (in_features, out_features); nn.Linear stores (out_features, in_features).
    in_features, out_features = conv.weight.shape
    linear = nn.Linear(in_features, out_features, bias=conv.bias is not None)
    with torch.no_grad():
        linear.weight.copy_(conv.weight.T)
        if conv.bias is not None:
            linear.bias.copy_(conv.bias)
    return linear


def replace_conv1d_with_linear(module: nn.Module) -> None:
    for name, child in list(module.named_children()):
        if isinstance(child, Conv1D):
            setattr(module, name, conv1d_to_linear(child).to(next(module.parameters()).device))
        else:
            replace_conv1d_with_linear(child)


replace_conv1d_with_linear(model)

# Sanity-check: the model still produces the same logits as before within fp tolerance.
with torch.no_grad():
    ids = tokenizer("The quick brown fox", return_tensors="pt").input_ids.to(device)
    logits = model(ids).logits
print("After conversion, logits shape:", tuple(logits.shape))
print(model.transformer.h[0].attn)  # c_attn / c_proj are now nn.Linear

After conversion, logits shape: (1, 4, 50257)
GPT2Attention(
  (c_attn): Linear(in_features=768, out_features=2304, bias=True)
  (c_proj): Linear(in_features=768, out_features=768, bias=True)
  (attn_dropout): Dropout(p=0.1, inplace=False)
  (resid_dropout): Dropout(p=0.1, inplace=False)
)


## 3 · Baseline generation (before fine-tuning)

So we have a "before" reference for the style transfer. Save these outputs — you'll diff them against the post-LoRA samples later.


In [ ]:
@torch.no_grad()
def generate(prompt: str, max_new_tokens: int = 80, temperature: float = 0.9, top_p: float = 0.95, seed: int = 0) -> str:
    torch.manual_seed(seed)
    model.eval()
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    out = model.generate(
        ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)


PROMPTS = [
    "ROMEO:",
    "To be, or not to be,",
    "Once upon a time in fair Verona,",
]

print("=== BASELINE (no fine-tuning) ===")
for p in PROMPTS:
    print("-" * 60)
    print(generate(p, seed=SEED))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


=== BASELINE (no fine-tuning) ===
------------------------------------------------------------
ROMEO: What you said that it was better, and it was good for me, to have my own way with my son. (Laughs.) That was good for him."

That's how you see him. It's how you see him.

I'm very proud of that, and I think I'm doing a lot better than he did. But at the same time, there are many
------------------------------------------------------------
To be, or not to be, for the time being, as the court does not have the authority to impose such a penalty, let us not forget the fact that the Court has no authority whatsoever to impose a fine under this act.

[Footnotes 1] Mr Justice Roberts is asked to rule on the issue of whether the imposition of a fine violates the Due Process Clause or the Fourteenth Amendment, but he fails to answer
------------------------------------------------------------
Once upon a time in fair Verona, we were able to learn something about our family.

"We grew up in t

## 4 · TODO: Implement `LoRALinear` (10 points)

Wrap an existing `nn.Linear` so the forward pass becomes:

$$
y = W_0 x + b + \frac{\alpha}{r} B A \cdot \text{dropout}(x)
$$

Constraints:

- The base layer's weight and bias must be **frozen** (`requires_grad = False`).
- $A$: shape `(r, in_features)`, initialised with `nn.init.kaiming_uniform_(a=math.sqrt(5))`.
- $B$: shape `(out_features, r)`, initialised with **zeros**. (Why zeros? See the background section.)
- Optional dropout on the input *before* it hits the LoRA branch.
- `merged_weight()` returns $W_0 + \frac{\alpha}{r} B A$ — useful for checking equivalence and for inference-time merging.

Fill in the `# TODO` lines.


In [ ]:
class LoRALinear(nn.Module):
    """Frozen linear layer + trainable low-rank residual."""

    def __init__(self, base: nn.Linear, r: int = 8, alpha: int = 16, dropout: float = 0.0):
        super().__init__()
        assert isinstance(base, nn.Linear), "LoRALinear only wraps nn.Linear in this homework."
        self.base = base
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        in_features = base.in_features
        out_features = base.out_features

        # TODO 4.1: freeze base weight & bias.
        # We ensure the original pre-trained weights are not updated.
        self.base.weight.requires_grad = False
        if self.base.bias is not None:
            self.base.bias.requires_grad = False

        # TODO 4.2: create the low-rank parameters.
        # Initialize A with kaiming_uniform_ and B with zeros.
        # Using base.weight.device/dtype ensures the adapter matches the model's location (e.g., GPU).
        self.lora_A = nn.Parameter(torch.empty((r, in_features),
                                              device=base.weight.device,
                                              dtype=base.weight.dtype))
        self.lora_B = nn.Parameter(torch.zeros((out_features, r),
                                               device=base.weight.device,
                                               dtype=base.weight.dtype))

        # A is random to allow gradients to flow; B is zero so the initial Delta W is 0.
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

        # TODO 4.3: dropout on input.
        self.lora_dropout = nn.Dropout(p=dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO 4.4: return base(x) + scaling * dropout(x) @ A.T @ B.T
        # We apply the frozen base weights and add the low-rank bypass result.
        base_output = self.base(x)
        lora_branch = (self.lora_dropout(x) @ self.lora_A.t() @ self.lora_B.t()) * self.scaling
        return base_output + lora_branch

    @torch.no_grad()
    def merged_weight(self) -> torch.Tensor:
        # TODO 4.5: return W0 + scaling * B @ A    (shape: out_features x in_features)
        # This is useful for inference to eliminate the latency of the LoRA branch.
        return self.base.weight + self.scaling * (self.lora_B @ self.lora_A)

In [ ]:
# Sanity check: at init, LoRALinear must be functionally identical to the base layer.
torch.manual_seed(0)
base = nn.Linear(64, 32).to(device)
lora = LoRALinear(base, r=4, alpha=8)  # NOTE: no trailing .to(device) -- LoRALinear should already place params on base's device.

assert lora.lora_A.device == base.weight.device, (
    f"lora_A on {lora.lora_A.device} but base on {base.weight.device}. "
    "Allocate lora_A/lora_B on base.weight.device inside __init__."
)
assert lora.lora_B.device == base.weight.device, "lora_B on wrong device, see hint in TODO 4.2."

x = torch.randn(2, 5, 64, device=device)
y_base = base(x)
y_lora = lora(x)

assert torch.allclose(y_base, y_lora, atol=1e-6), "LoRA forward differs from base at init — B should be zeros!"
print("OK: LoRALinear matches base at init.")

# After perturbing B, outputs should differ.
with torch.no_grad():
    lora.lora_B.add_(torch.randn_like(lora.lora_B))
assert not torch.allclose(base(x), lora(x), atol=1e-6)
print("OK: LoRALinear differs from base after perturbing B.")

# Merged weight equivalence.
with torch.no_grad():
    merged = lora.merged_weight()
    y_merged = F.linear(x, merged, base.bias)
    assert torch.allclose(y_merged, lora(x), atol=1e-5), "merged_weight() inconsistent with forward()."
print("OK: merged_weight() matches forward().")

OK: LoRALinear matches base at init.
OK: LoRALinear differs from base after perturbing B.
OK: merged_weight() matches forward().


## 5 · TODO: Inject LoRA into GPT-2 and freeze the rest (5 points)

You'll write `inject_lora` that walks the model, finds every `nn.Linear` whose attribute name matches `target_names`, and replaces it with a `LoRALinear` wrapping it.

We'll target `("c_attn", "c_proj")`:

- `c_attn` — the fused QKV projection (one per block, in `attn`).
- `c_proj` — *appears in two places* per block: the attention output projection (`attn.c_proj`) **and** the MLP down-projection (`mlp.c_proj`). Matching by attribute name will hit both — that's fine and is consistent with what `peft` does by default (it suffix-matches the same names). It's a useful gotcha to be aware of.

So you should end up with **36 wrappings** = 12 blocks × (1 `c_attn` + 1 `attn.c_proj` + 1 `mlp.c_proj`).

After injection, **only `lora_A` and `lora_B` should have `requires_grad=True`.**


In [ ]:
def inject_lora(
    module: nn.Module,
    target_names: tuple[str, ...] = ("c_attn", "c_proj"),
    r: int = 8,
    alpha: int = 16,
    dropout: float = 0.05,
) -> int:
    """Recursively swap nn.Linear children whose attribute name is in target_names."""
    n_wrapped = 0

    # We use named_children() to get direct sub-modules (like 'transformer', 'h', etc.)
    for name, child in module.named_children():
        # Check if this child is a Linear layer we want to target
        if isinstance(child, nn.Linear) and name in target_names:
            # TODO 5.1: Replace the child with our LoRALinear wrapper
            # setattr allows us to dynamically overwrite the attribute on the module
            new_layer = LoRALinear(child, r=r, alpha=alpha, dropout=dropout)
            setattr(module, name, new_layer)
            n_wrapped += 1
        else:
            # If it's not a match, we must look inside its children (recursion)
            # This allows us to reach deep into the 12 blocks of GPT-2
            n_wrapped += inject_lora(child, target_names, r, alpha, dropout)

    return n_wrapped


def freeze_non_lora(model: nn.Module) -> None:
    """Sets requires_grad=False on every parameter whose name does not contain 'lora_'."""
    # TODO 5.2
    for name, param in model.named_parameters():
        if "lora_" not in name:
            param.requires_grad = False

def count_params(model: nn.Module) -> tuple[int, int]:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


def assert_same_device(model: nn.Module) -> None:
    """Sanity check: every parameter lives on the same device. Catches a common LoRA bug."""
    expected = next(model.parameters()).device
    for n, p in model.named_parameters():
        assert p.device == expected, f"param {n} on {p.device}, expected {expected}"

In [ ]:
# Reload a fresh GPT-2 (we'll keep the converted Linear version so LoRALinear can wrap it).
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)
replace_conv1d_with_linear(model)

n_wrapped = inject_lora(model, target_names=("c_attn", "c_proj"), r=8, alpha=16, dropout=0.05)
freeze_non_lora(model)

trainable, total = count_params(model)
print(f"Wrapped {n_wrapped} layers")
print(f"Trainable: {trainable/1e6:.3f}M  /  Total: {total/1e6:.1f}M  ({100*trainable/total:.2f}%)")
assert n_wrapped == 36, (
    f"Expected 36 (12 blocks x (c_attn + attn.c_proj + mlp.c_proj)), got {n_wrapped}"
)
assert trainable < 1_500_000, "Way too many trainable params — check freeze_non_lora."
assert_same_device(model)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Wrapped 36 layers
Trainable: 0.811M  /  Total: 125.3M  (0.65%)


**Q1. (5 points)** What fraction of the original 124M parameters did you make trainable? With $r=8$ and target modules `(c_attn, c_proj)`, you should be around 0.6–0.7%. Think about why: how many params does each LoRA pair add for a `(d_in, d_out)` linear, and what are the dims of the three target layer types?

By targeting the attention and MLP projection layers with a rank of 8, we only need to train about 0.65% of the total model weights. This significantly reduces the memory and compute required.


## Per-pair formula
- For a `(d_in, d_out)` linear, a single LoRA pair (A of shape `(r, d_in)`, B of shape `(d_out, r)`) adds **r×(din+dout)** parameters.

This formula accounts for matrix $A \in \mathbb{R}^{r \times d_{in}}$ and matrix $B \in \mathbb{R}^{d_{out} \times r}$.

## Per-layer LoRA params at r=8
- `attn.c_attn`  (768 → 2304): **24,576** params

    Calculation: $8 \times (768 + 2304) = 8 \times 3072$
- `attn.c_proj`  (768 →  768): **12,288** params

    Calculation: $8 \times (768 + 768) = 8 \times 1536$

- `mlp.c_proj`   (3072 → 768): **30,720** params

    Calculation: $8 \times (3072 + 768) = 8 \times 3840$

## Totals
- Trainable parameters across all 12 blocks: 811,008 params
  Sum of the three layers above: $24,576 + 12,288 + 30,720 = 67,584$

  Calculation: $12 \text{ blocks} \times 67,584 \text{ params/block}$


- Fraction of GPT-2 small (124M): **0.654 %**

  Calculation: $(811,008 / 124,000,000) \times 100$



## 6 · Dataset: TinyShakespeare

A single ~1MB text file. We tokenize the whole thing once, then chunk into fixed-length blocks.


In [ ]:
import urllib.request

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
TINY_SHAKES_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
TINY_SHAKES_PATH = os.path.join(DATA_DIR, "tinyshakespeare.txt")
if not os.path.exists(TINY_SHAKES_PATH):
    urllib.request.urlretrieve(TINY_SHAKES_URL, TINY_SHAKES_PATH)

with open(TINY_SHAKES_PATH) as f:
    raw_text = f.read()

print(f"Corpus: {len(raw_text):,} chars")
print(raw_text[:300])

Corpus: 1,115,394 chars
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


In [ ]:
BLOCK_SIZE = 256


class LMChunks(Dataset):
    """Tokenize a long string once, then yield contiguous fixed-length chunks for next-token LM training."""

    def __init__(self, text: str, tokenizer, block_size: int):
        self.block_size = block_size
        ids = tokenizer(text, return_tensors="pt").input_ids[0]
        # Drop the tail so we have a clean multiple of block_size.
        n_chunks = ids.size(0) // block_size
        self.ids = ids[: n_chunks * block_size].view(n_chunks, block_size)

    def __len__(self) -> int:
        return self.ids.size(0)

    def __getitem__(self, idx: int) -> dict:
        chunk = self.ids[idx]
        # For causal LM, labels = input_ids; HF will internally shift by one.
        return {"input_ids": chunk, "labels": chunk.clone()}


full_ds = LMChunks(raw_text, tokenizer, BLOCK_SIZE)
n_val = max(1, len(full_ds) // 20)
train_ds, val_ds = torch.utils.data.random_split(
    full_ds, [len(full_ds) - n_val, n_val], generator=torch.Generator().manual_seed(SEED)
)
print(f"Train chunks: {len(train_ds)} | Val chunks: {len(val_ds)} | block_size={BLOCK_SIZE}")

Token indices sequence length is longer than the specified maximum sequence length for this model (338025 > 1024). Running this sequence through the model will result in indexing errors


Train chunks: 1254 | Val chunks: 66 | block_size=256


## 6.5 · Evaluation harness: perplexity

We need a quantitative sanity check, not just vibes from generated samples. Two metrics:

1. **Shakespeare-val PPL** — should drop sharply after fine-tuning (the model is getting better at predicting Shakespeare tokens).
2. **General-English PPL** — computed on a held-out non-Shakespeare snippet (Pride & Prejudice). This catches **catastrophic forgetting**: if it skyrockets, your LoRA has destroyed general English ability in chasing Shakespeare style. Mild rises are normal; large rises mean your `r`/`alpha`/lr is too aggressive or you're training too long.

> Note: right now `model` already has LoRA injected with $B = 0$, so its outputs are mathematically identical to plain GPT-2. The "baseline" PPL we print below is therefore the un-tuned GPT-2's PPL on each corpus.


MO NOTES:
What is **Perplexity (PPL)**?
is a measure of how "surprised" a model is by a sequence of text.

**Low Perplexity**: The model finds the text very predictable. It "understands" the patterns, vocabulary, and style.

**High Perplexity**: The model is confused. The text contains words or structures it didn't expect.

Mathematically, it is the exponent of the cross-entropy loss ($e^{loss}$). If your training loss is 4.5, your perplexity is $e^{4.5} \approx 90$. As the model learns Shakespeare, the loss will go down, and the PPL will drop significantly.

MO NOTES: **Catastrophic Forgetting**

This is the term for when a neural network "forgets" the information it was originally trained on (General English) because it is learning a new task (Shakespeare) too aggressively.

LoRA is naturally very good at preventing catastrophic forgetting because it only updates **a tiny fraction** of the weights ($<1\%$), leaving the "core knowledge" of GPT-2 frozen and untouched.

In [ ]:
def collate(batch):
    return {k: torch.stack([b[k] for b in batch], dim=0) for k in batch[0]}


@torch.no_grad()
def compute_ppl(model: nn.Module, dataset, batch_size: int = 8) -> float:
    """Token-weighted perplexity over a chunk dataset. Lower = better."""
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate)
    total_loss = 0.0
    total_tokens = 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        # HF computes mean cross-entropy over (B*T) shifted-label positions; weight by that token count.
        n_tok = batch["labels"].numel()
        total_loss += out.loss.item() * n_tok
        total_tokens += n_tok
    return math.exp(total_loss / total_tokens)

In [ ]:
# Forgetting-control corpus: a public-domain Pride & Prejudice excerpt (no Shakespeare in sight).
control_text = """It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife. However little known the feelings or views of such a man may be on his first entering a neighbourhood, this truth is so well fixed in the minds of the surrounding families, that he is considered as the rightful property of some one or other of their daughters.

"My dear Mr. Bennet," said his lady to him one day, "have you heard that Netherfield Park is let at last?"

Mr. Bennet replied that he had not.

"But it is," returned she; "for Mrs. Long has just been here, and she told me all about it."

Mr. Bennet made no answer.

"Do not you want to know who has taken it?" cried his wife impatiently.

"You want to tell me, and I have no objection to hearing it."

This was invitation enough.

"Why, my dear, you must know, Mrs. Long says that Netherfield is taken by a young man of large fortune from the north of England; that he came down on Monday in a chaise and four to see the place, and was so much delighted with it that he agreed with Mr. Morris immediately; that he is to take possession before Michaelmas, and some of his servants are to be in the house by the end of next week."

"What is his name?"

"Bingley."

"Is he married or single?"

"Oh! single, my dear, to be sure! A single man of large fortune; four or five thousand a year. What a fine thing for our girls!"

"How so? how can it affect them?"

"My dear Mr. Bennet," replied his wife, "how can you be so tiresome! You must know that I am thinking of his marrying one of them."

"Is that his design in settling here?"

"Design! nonsense, how can you talk so! But it is very likely that he may fall in love with one of them, and therefore you must visit him as soon as he comes."
"""

control_ds = LMChunks(control_text, tokenizer, BLOCK_SIZE)
print(f"Control chunks: {len(control_ds)} (block_size={BLOCK_SIZE})")
assert len(control_ds) >= 1, "Control corpus too short — add more text."

Control chunks: 1 (block_size=256)


MO NOTES: Why is there only 1 chunk?


1,500 characters produce roughly 350–400 tokens. BLOCK_SIZE is 256.

The code takes the first 256 tokens to create one clean chunk and throws away the remaining ~144 tokens because they aren't enough to make a full second block.

In [ ]:
ppl_shakes_before = compute_ppl(model, val_ds)
ppl_control_before = compute_ppl(model, control_ds)
print(f"Baseline Shakespeare-val PPL : {ppl_shakes_before:7.2f}")
print(f"Baseline Control      PPL    : {ppl_control_before:7.2f}")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Baseline Shakespeare-val PPL :   93.53
Baseline Control      PPL    :   16.92


## 7 · Training loop

The loop itself is given. **TODO 7.1**: build the optimizer over only the trainable parameters.


In [ ]:
@dataclass
class TrainCfg:
    epochs: int = 2
    batch_size: int = 8
    lr: float = 3e-4
    weight_decay: float = 0.0
    warmup_ratio: float = 0.05
    log_every: int = 50
    grad_clip: float = 1.0


def train_lora(model: nn.Module, train_ds, val_ds, cfg: TrainCfg) -> dict:
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate)

    # TODO 7.1: build an AdamW over only the parameters with requires_grad=True.
    # (Passing all params still trains correctly because frozen ones have no grads,
    # but it wastes optimizer-state memory — a real cost on bigger models.)
    optim = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=cfg.lr,
            weight_decay=cfg.weight_decay
        )
    total_steps = len(train_loader) * cfg.epochs
    warmup_steps = int(cfg.warmup_ratio * total_steps)

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.0, 0.5 * (1 + math.cos(math.pi * progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(optim, lr_lambda)

    history = {"train_loss": [], "val_loss": []}
    step = 0
    for epoch in range(cfg.epochs):
        model.train()
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            loss = out.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg.grad_clip)
            optim.step()
            sched.step()
            optim.zero_grad(set_to_none=True)

            if step % cfg.log_every == 0:
                history["train_loss"].append((step, loss.item()))
                print(f"epoch {epoch} step {step:4d} | train loss {loss.item():.4f} | lr {sched.get_last_lr()[0]:.2e}")
            step += 1

        # validation
        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                val_losses.append(model(**batch).loss.item())
        v = sum(val_losses) / len(val_losses)
        history["val_loss"].append((step, v))
        print(f"== epoch {epoch} val loss {v:.4f} (ppl {math.exp(v):.2f}) ==")

    return history


cfg = TrainCfg(epochs=2, batch_size=8, lr=3e-4)
history = train_lora(model, train_ds, val_ds, cfg)

epoch 0 step    0 | train loss 4.4137 | lr 2.00e-05
epoch 0 step   50 | train loss 4.2209 | lr 2.89e-04
epoch 0 step  100 | train loss 3.8991 | lr 2.43e-04
epoch 0 step  150 | train loss 3.5237 | lr 1.71e-04
== epoch 0 val loss 3.7285 (ppl 41.62) ==
epoch 1 step  200 | train loss 3.9718 | lr 9.39e-05
epoch 1 step  250 | train loss 3.6636 | lr 3.17e-05
epoch 1 step  300 | train loss 4.0026 | lr 1.40e-06
== epoch 1 val loss 3.7055 (ppl 40.67) ==


**MO Notes:**
Optimizers like AdamW don't just multiply gradients by the learning rate. They **keep a "running history"** (momentum and variance) for every single parameter you give them.

If we passed model.parameters() without filtering, AdamW would allocate memory to track the history of all 124 million GPT-2 weights. This would consume an extra ~1 Gigabyte of GPU memory just for the optimizer state, even though those weights are frozen!

By adding the **if p.requires_grad filter**, AdamW only tracks the history for the ~811,000 LoRA parameters we injected. This uses almost zero memory, which is the entire reason LoRA can be trained on small, cheap GPUs.

**MO NOTES:**
**The Learning Rate "Rollercoaster" (Warmup + Cosine Decay)**


In the logs, the lr starts tiny (2.00e-05), hits a peak (2.89e-04), and then drops significantly by step 300 (1.40e-06). This is controlled by lr_lambda function.

**Warmup (step < warmup_steps):** For the first 5% of training, the learning rate ramps up from zero. This prevents the model from "panicking" and making massive, destructive weight updates when it first sees the new Shakespeare data.

Cosine Decay: After the warmup, the learning rate follows a cosine curve down to zero. This helps the model "settle" into the best possible version of the weights as it nears the end of training.

**MO NOTES:**

2 epochs is likely enough. In the first epoch, you made a massive leap **from 93 down to 41**. In the second epoch, you only moved from **41 to 40**. This is a classic **plateau**. The model has already learned the most obvious "Shakespearean" patterns (the vocabulary, the character names, the colons). Adding a 3rd or 4th epoch might only drop the PPL by a tiny fraction while taking up more time and GPU resources.

The Risk of Overfitting -

If we continue training too long, the model might stop learning the style of Shakespeare and start memorizing the specific lines in the 1MB tinyshakespeare .txt file

## 8 · Generate after fine-tuning

Same prompts, same seed. Read the outputs and compare to the baseline.


In [ ]:
print("=== AFTER LoRA FINE-TUNE ===")
for p in PROMPTS:
    print("-" * 60)
    print(generate(p, seed=SEED))


=== AFTER LoRA FINE-TUNE ===
------------------------------------------------------------
ROMEO:
How long that time will be, and how it will be, and how it will be.

LUKE ROBERTSON:
Well, what is it,
that you see him, my lord?

LUKE ROBERTSON:
Truly, that I saw.

ROMEO:
How will it be, your lord?

LUKE
------------------------------------------------------------
To be, or not to be, for you?

RICHARD:
It is hard, and it is not at all pleasant to be a man.

MENINIUS:
But, my lord, it is indeed so.

RICHARD:
I am very glad, and very glad.

MENINIUS:
How will you thank us?

RICHARD:
------------------------------------------------------------
Once upon a time in fair Verona,
She beheld the prince, and called him lord.

ROME:
How did he not do it?

BRIAN:
Hench's sister
And his father's, my lord!

ROME:
Wherefore, how good and noble
How, how glorious are these princes:
How will they be at your hands:
Take him for


In [ ]:
ppl_shakes_after = compute_ppl(model, val_ds)
ppl_control_after = compute_ppl(model, control_ds)
print(f"{'metric':<30}{'before':>10}{'after':>10}{'delta':>10}")
print(f"{'Shakespeare-val PPL':<30}{ppl_shakes_before:>10.2f}{ppl_shakes_after:>10.2f}{ppl_shakes_after - ppl_shakes_before:>+10.2f}")
print(f"{'Control (P&P) PPL':<30}{ppl_control_before:>10.2f}{ppl_control_after:>10.2f}{ppl_control_after - ppl_control_before:>+10.2f}")

metric                            before     after     delta
Shakespeare-val PPL                93.53     41.68    -51.85
Control (P&P) PPL                  16.92     18.51     +1.59


**Q2. (5 points)** Qualitatively, how did the outputs change? Mention at least two specific shifts you can point to (vocabulary, punctuation/structure, character names, line breaks, etc.).

`# YOUR ANSWER:`

## Stylistic shifts after fine-tuning
- Shift 1 (vocabulary / diction): #YOUR ANSWER HERE#
- Shift 2 (formatting / structure): #YOUR ANSWER HERE#
- Shift 3 (optional, e.g. named entities): #YOUR ANSWER HERE#


Q2 Answer: Stylistic Shifts after Fine-Tuning
Comparing your Baseline (which sounds like modern prose or legal text) to your After LoRA samples, here are the primary changes:

## Stylistic shifts after fine-tuning

**Shift 1 (Vocabulary / Diction):** The model shifted from modern, mundane language (e.g., "my own way with my son," "legal authority," "San Gabriel") to Elizabethan-style vocabulary and honorifics. Specifically, it now frequently uses the term "my lord" and words like "beheld," "wherefore," "noble," and "glorious."

**Shift 2 (Formatting / Structure)**: The baseline output was structured in standard prose paragraphs and conversational blocks. After fine-tuning, the model adopted a dramatic script format. It now places character names in all-caps followed by a colon on their own line (e.g., RICHARD:, MENINIUS:) and uses much shorter, verse-like line breaks instead of long sentences.

**Shift 3 (Optional - Named Entities):** While the baseline model talked about generic "parents" or specific modern locations like "Hilo," the fine-tuned model immediately introduced names found in the Shakespearean corpus, such as Richard and Menenius (a character from Coriolanus).

Why did this happen?
Before fine-tuning, GPT-2 was predicting the "next token" based on its knowledge of the internet (Wikipedia, news, Reddit). After you ran the training loop, the LoRA layers learned the specific statistical patterns of the TinyShakespeare dataset.

Even though the model's "factual" knowledge of the world is still in the frozen weights, the LoRA adapter has successfully "steered" the model's output toward the syntax (structure) and lexicon (vocabulary) of a 16th-century play.

Why did we see "Luke Robertson"?
This could been an example of of a "hallucination" what happens when the Frozen Weights and the LoRA Weights disagree. The model knows that a play should have a name in all caps (used LoRA), and it’s mixing its original knowledge with the new format it learned. The Frozen Weights (GPT-2's original brain) had a very strong memory of a name like "LUKE ROBERTSON" from its internet training.

**Q3. (5 points)** Report the four PPL numbers above. Did the control PPL move? In which direction, and by how much relative to the Shakespeare drop? What does this tell you about the trade-off you're making with LoRA fine-tuning?

`# YOUR ANSWER:`

## PPL numbers
| metric              | before  | after   |
|---------------------|--------:|--------:|
| Shakespeare-val PPL |93.53 | 41.68 |
| Control (P&P) PPL   | 16.92 | 18.51  |

93.53     41.68    -51.85
Control (P&P) PPL                  16.92     18.51     +1.59

## Direction and magnitude
- Control PPL moved (up / down / unchanged): up +1.59
- Magnitude relative to the Shakespeare-PPL drop (smaller / similar / larger): smaller -51.85

## Trade-off interpretation
This result illustrated the trade-off known as Catastrophic Forgetting. When we are training a model on a specific style (Shakespeare), we are slightly "overfitting" general language capabilities. However, this negative effect is minimal because we use LoRA. 99% weights remain frozen. And we see the huge drop in Shakespeare PPL. We add new skills and do not destroy old ones.



## 9 · TODO: Save and load just the adapter (5 points)

A LoRA adapter is tiny (megabytes). Below, save **only** the LoRA parameters to disk, then verify you can load them back into a fresh GPT-2 and reproduce the trained model's outputs.


In [ ]:
ADAPTER_PATH = "lora_adapter.pt"


import os
import torch
import torch.nn as nn

def save_adapter(model: nn.Module, path: str) -> None:
    # TODO 9.1: collect a state_dict containing only parameters whose name contains "lora_".
    # We iterate over the full state_dict and filter it down.
    lora_state_dict = {name: tensor for name, tensor in model.state_dict().items() if "lora_" in name}

    # Save only the filtered dictionary
    torch.save(lora_state_dict, path)


def load_adapter(model: nn.Module, path: str) -> None:
    # TODO 9.2: torch.load and copy the saved tensors into the matching parameters of model.
    # weights_only=True is a modern PyTorch security best practice when loading files
    adapter_state_dict = torch.load(path, weights_only=True)

    # We MUST use strict=False.
    # Why? Because our adapter_state_dict only contains a few LoRA weights,
    # while the 'model' contains the full 124M GPT-2 weights. strict=True would
    # crash because it expects every single GPT-2 weight to be in the file.
    result = model.load_state_dict(adapter_state_dict, strict=False)

    # Optional sanity check to fulfill the "every saved key must exist" constraint
    if len(result.unexpected_keys) > 0:
        raise RuntimeError(f"Found unexpected keys in the adapter file: {result.unexpected_keys}")


save_adapter(model, ADAPTER_PATH)
print(f"Adapter file size: {os.path.getsize(ADAPTER_PATH) / 1e6:.2f} MB")

Adapter file size: 3.27 MB


In [ ]:
# Round-trip check: build a fresh GPT-2 + LoRA, load the adapter, confirm outputs match.
fresh = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)
replace_conv1d_with_linear(fresh)
inject_lora(fresh, target_names=("c_attn", "c_proj"), r=8, alpha=16, dropout=0.05)
freeze_non_lora(fresh)
load_adapter(fresh, ADAPTER_PATH)
assert_same_device(fresh)

# IMPORTANT: a freshly-constructed nn.Module defaults to train mode, which leaves
# both GPT-2's own dropouts (attn_pdrop, resid_pdrop) and LoRALinear's dropout
# active. torch.no_grad() turns off autograd but NOT dropout — that's controlled
# by module.training. So set both models to eval before comparing logits.
model.eval()
fresh.eval()

with torch.no_grad():
    ids = tokenizer("ROMEO:", return_tensors="pt").input_ids.to(device)
    a = model(ids).logits
    b = fresh(ids).logits

print("max abs diff:", (a - b).abs().max().item())
assert torch.allclose(a, b, atol=1e-5), "Adapter round-trip failed."
print("OK: adapter round-trips.")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


max abs diff: 0.0
OK: adapter round-trips.


## 10 · TODO: Compare with the `peft` library (5 points)

Re-run the same fine-tune using `peft.LoraConfig` + `get_peft_model` on a fresh GPT-2 (with the original `Conv1D` layers — `peft` knows how to handle them). Match the hyperparameters: `r=8`, `alpha=16`, `dropout=0.05`, target modules `c_attn` and `c_proj`.


MO NOTES:


**PEFT** stands for **Parameter-Efficient Fine-Tuning**. It is an open-source library created by Hugging Face to standardize how we adapt massive AI models (like Llama, GPT, or Stable Diffusion) without needing to train all their billions of parameters.

The peft library supports many techniques as well, such as:

- LoRA

- Prefix Tuning: Adding trainable tensors to the beginning of hidden states.  


- Prompt Tuning: Learning a specialized "prompt" vector.  

- IA3: Scaling intermediate activations.

In [ ]:
!pip install --upgrade "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 76.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

peft_base = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)

# TODO 10.1: build a LoraConfig matching your hand-rolled setup.
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["c_attn", "c_proj"],  # Matches the layers we targeted earlier
    bias="none",
)
peft_model = get_peft_model(peft_base, peft_config).to(device)  # belt-and-suspenders: ensure adapter weights land on GPU
peft_model.print_trainable_parameters()
assert_same_device(peft_model)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable params: 811,008 || all params: 125,250,816 || trainable%: 0.6475


In [ ]:
# TODO 10.2: train peft_model with the same TrainCfg you used above.
# Tip: peft_model behaves like a normal nn.Module; the train_lora() function works as-is.
peft_history = train_lora(peft_model, train_ds, val_ds, cfg)

epoch 0 step    0 | train loss 4.6584 | lr 2.00e-05
epoch 0 step   50 | train loss 3.9816 | lr 2.89e-04
epoch 0 step  100 | train loss 3.6552 | lr 2.43e-04
epoch 0 step  150 | train loss 3.9219 | lr 1.71e-04
== epoch 0 val loss 3.7283 (ppl 41.61) ==
epoch 1 step  200 | train loss 3.4448 | lr 9.39e-05
epoch 1 step  250 | train loss 4.1064 | lr 3.17e-05
epoch 1 step  300 | train loss 3.6054 | lr 1.40e-06
== epoch 1 val loss 3.7041 (ppl 40.61) ==


In [ ]:
# Generate from the PEFT-trained model with the same prompts/seed.
print("=== PEFT-trained ===")
backup = model
model = peft_model  # so generate() uses it
for p in PROMPTS:
    print("-" * 60)
    print(generate(p, seed=SEED))
model = backup

=== PEFT-trained ===
------------------------------------------------------------
ROMEO:
How long that time will be, and how it will be, and how it will be.

LUKE ROBERTSON:
Well, what is it,
that you see him, my lord?

LUKE ROBERTSON:
Truly, that I saw.

ROMEO:
How will it be, your lord?

LUKE
------------------------------------------------------------
To be, or not to be, for you?

MEENESTO:
I have been, but not so much as I could do; for I will, and must, and will not;

BUDDY:
Now, I will, and must, and will not;
For what else will I find than to go to war?

MEENESTO:
Let there be more
------------------------------------------------------------
Once upon a time in fair Verona,
She beheld the prince, and called him lord.

RICHARDIUS:
To-night, be ready, and meet for a battle,
That is not so much a battle, as a quarrel.

LUCIUS:
Treat him, then, with patience.

RICHARDIUS:
To-night, be in his


In [ ]:
# Quantitative comparison: PEFT vs hand-rolled, on the same val/control sets.
ppl_shakes_peft = compute_ppl(peft_model, val_ds)
ppl_control_peft = compute_ppl(peft_model, control_ds)
print(f"{'metric':<30}{'hand-rolled':>14}{'PEFT':>10}")
print(f"{'Shakespeare-val PPL':<30}{ppl_shakes_after:>14.2f}{ppl_shakes_peft:>10.2f}")
print(f"{'Control (P&P) PPL':<30}{ppl_control_after:>14.2f}{ppl_control_peft:>10.2f}")

metric                           hand-rolled      PEFT
Shakespeare-val PPL                    41.68     41.64
Control (P&P) PPL                      18.51     18.56


**Q4.(5 points)** Compare your hand-rolled trainable parameter count vs `peft_model.print_trainable_parameters()`. Are they identical? If not, where does the difference come from? (Hint: think about which projections `peft` decides to wrap given `target_modules=["c_attn", "c_proj"]`.)

## Param counts

Hand-rolled trainable params: 811,008

PEFT trainable params: 811,008

Identical? (yes / no): Yes

## Where any difference comes from (or why they match):

The counts are identical because both implementations target the exact same 36 linear projections within the GPT-2 architecture.

In both cases, we specified target_modules=["c_attn", "c_proj"]. This resulted in the injection of LoRA adapters into:

c_attn: The fused Query-Key-Value projection ($768 \rightarrow 2304$) in each of the 12 blocks.

attn.c_proj: The attention output projection ($768 \rightarrow 768$) in each of the 12 blocks.

mlp.c_proj: The MLP down-projection ($3072 \rightarrow 768$) in each of the 12 blocks.

Even though the "hand-rolled" version converted the model to nn.Linear first while peft wrapped the original Conv1D layers, the input and output dimensions remained exactly the same.
Since the LoRA parameter count is determined solely by the rank ($r=8$) and those dimensions—using the formula $r \times (d_{in} + d_{out})$—the total number of added trainable parameters is identical.

**Q5.(5 points)** After the same number of training steps, did your hand-rolled LoRA reach a similar Shakespeare-val PPL to the PEFT version? List one reason they could differ even with matched hyperparameters.

## Shakespeare-val PPL comparison
- Hand-rolled: **41.68**
- PEFT: **41.64**
- Within ~10% of each other? (yes / no): Yes (they are within ~0.1% of each other)

## One reason they could still differ with matched hyperparameters:

The most likely reason for the tiny numerical difference is floating-point precision and the order of operations.

In the hand-rolled version, we manually converted GPT-2’s Conv1D layers into standard nn.Linear layers by transposing the weights before injecting LoRA. In contrast, the peft library wraps the original Conv1D layers directly. Even though these are mathematically equivalent, the underlying GPU kernels (the math operations performed by the hardware) differ.

Small variations in the order of additions and multiplications in these kernels lead to tiny floating-point errors (epsilon differences). Because training is an iterative process where every step depends on the last, these "epsilon" differences accumulate over hundreds of steps, leading to the slight divergence you see in the final PPL.


## 11 · Bonus: pick a different style

Pick one (or all three!) and watch the model become a different character. Each option below builds a `raw_text` string; everything downstream of `LMChunks(raw_text, …)` is identical.

> **Tip:** for any of these, **reset the model** before the new fine-tune (re-run cell 5 first), or you'll be training a Shakespeare-flavoured Rick. Or do that on purpose. It's funny.


### 11A · Rick & Morty

A community dialogue dataset is hosted on the Hugging Face Hub. Substring filter to Rick's lines for stronger style transfer.


In [ ]:
# Bonus A: Rick & Morty dialogue.
# This uses the `Prarabdha/Rick_and_Morty_Transcript` dataset on HF Hub.
# If that dataset is unavailable, swap in any other dialogue source — what matters is the format.
from datasets import load_dataset

try:
    ds = load_dataset("Prarabdha/Rick_and_Morty_Transcript", split="train")
    # Inspect: print(ds.column_names, ds[0])
    # Filter to Rick's lines and concatenate.
    rick_lines = [row["dialouge"] for row in ds if row.get("speaker", "").strip().lower() == "rick"]
    raw_text_rick = "\n".join(f"Rick: {line}" for line in rick_lines if isinstance(line, str))
    print(f"Rick corpus: {len(raw_text_rick):,} chars over {len(rick_lines)} lines")
    print(raw_text_rick[:400])
except Exception as e:
    print("Dataset load failed:", e)
    print("Fallback: paste your own transcript text into raw_text_rick.")
    raw_text_rick = ""

# To use this corpus, set `raw_text = raw_text_rick` and re-run the dataset / training cells.

README.md:   0%|          | 0.00/793 [00:00<?, ?B/s]

Rick-n-Morty.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/9618 [00:00<?, ? examples/s]

Dataset load failed: 'NoneType' object has no attribute 'strip'
Fallback: paste your own transcript text into raw_text_rick.


### 11B · Yoda

Yoda has a small but iconic corpus. We ship ~50 quotes inline as a starter — augment with your own scraping for stronger transfer (e.g., parse Star Wars scripts on imsdb.com).


In [ ]:
# Bonus B: Yoda. Inline starter corpus — feel free to extend.
yoda_quotes = [
    "Do or do not. There is no try.",
    "Fear is the path to the dark side. Fear leads to anger. Anger leads to hate. Hate leads to suffering.",
    "Size matters not. Look at me. Judge me by my size, do you?",
    "When nine hundred years old you reach, look as good you will not.",
    "Wars not make one great.",
    "Adventure. Heh. Excitement. Heh. A Jedi craves not these things.",
    "Patience you must have, my young padawan.",
    "Truly wonderful, the mind of a child is.",
    "Always pass on what you have learned.",
    "Train yourself to let go of everything you fear to lose.",
    "Difficult to see. Always in motion is the future.",
    "The greatest teacher, failure is.",
    "Pass on what you have learned. Strength. Mastery.",
    "Named must your fear be before banish it you can.",
    "Powerful you have become, the dark side I sense in you.",
    "You must unlearn what you have learned.",
    "That is why you fail.",
    "Once you start down the dark path, forever will it dominate your destiny.",
    "Mind what you have learned. Save you it can.",
    "Luminous beings are we, not this crude matter.",
    "Through the Force, things you will see. Other places. The future, the past. Old friends long gone.",
    "Decide you must, how to serve them best.",
    "Hard to see, the dark side is.",
    "Strong with the Force, young Skywalker is.",
    "Many of the truths that we cling to depend on our point of view.",
    "Smaller in number are we, but larger in mind.",
    "Around the survivors a perimeter create.",
    "Rejoice for those around you who transform into the Force.",
    "Mourn them, do not. Miss them, do not.",
    "When you look at the dark side, careful you must be.",
    "Help you I can. Yes. Mmm.",
    "Already know you that which you need.",
    "A Jedi's strength flows from the Force.",
    "Anger, fear, aggression. The dark side are they.",
    "If once you start down the dark path, forever will it dominate your destiny.",
    "Consume you it will, as it did Obi-Wan's apprentice.",
    "Do not underestimate the powers of the Emperor or suffer your father's fate, you will.",
    "Reckless he is. Matters are worse.",
    "Control, control. You must learn control.",
    "Yes, a Jedi's strength flows from the Force. But beware. Anger, fear, aggression.",
    "Stopped they must be. On this all depends.",
    "Faith in your friends, yours is.",
    "Soon will I rest. Yes, forever sleep. Earned it I have.",
    "Twilight is upon me, and soon night must fall.",
    "When gone am I, the last of the Jedi will you be.",
    "The Force runs strong in your family. Pass on what you have learned.",
    "There is another. Sky-walker.",
    "No. Try not. Do, or do not. There is no try.",
    "Judge me by my size, do you?",
    "Concentrate. Feel the Force flow.",
]

# Repeat to give the model enough tokens to chunk into BLOCK_SIZE windows.
raw_text_yoda = ("\n".join(yoda_quotes) + "\n") * 200
print(f"Yoda corpus: {len(raw_text_yoda):,} chars (after replication)")
print(raw_text_yoda[:300])


Yoda corpus: 495,200 chars (after replication)
Do or do not. There is no try.
Fear is the path to the dark side. Fear leads to anger. Anger leads to hate. Hate leads to suffering.
Size matters not. Look at me. Judge me by my size, do you?
When nine hundred years old you reach, look as good you will not.
Wars not make one great.
Adventure. Heh. E


**Q6 (bonus).** Pick at least one of A/B/C, fine-tune, and paste 3 generated samples that show the style. Also report Shakespeare-val PPL and control PPL after this run — does Shakespeare PPL get worse (it should: you're tuning toward a different style now)? One sentence on what surprised you most.

`# YOUR ANSWER:`

## Bonus run setup
- Dataset chosen (A=Rick & Morty, B=Hermione, C=Yoda): #YOUR ANSWER HERE#

## Three generated samples that show the style
1. #YOUR ANSWER HERE#
2. #YOUR ANSWER HERE#
3. #YOUR ANSWER HERE#

## PPL after the bonus fine-tune
- Shakespeare-val PPL: #YOUR ANSWER HERE#
- Control (P&P) PPL: #YOUR ANSWER HERE#
- Did Shakespeare PPL get worse vs the previous run? (yes / no): #YOUR ANSWER HERE#

## Most surprising observation (one sentence)
 #YOUR ANSWER HERE#


## 12 · Submission checklist

- [ ] All cells run top to bottom without error.
- [ ] Q1–Q5 answered. Q6 if you did the bonus.
- [ ] Baseline and post-fine-tune sample outputs are visible in the saved notebook.
- [ ] All four PPL numbers (Shakespeare/control × before/after) are printed.
- [ ] `lora_adapter.pt` round-trip assertion passes.
- [ ] Hand-rolled vs PEFT comparison runs and PPLs are reported side-by-side.

Good luck — and may the gradients flow strong with you.
